# KoMaP 2026 — Colab 실행

로컬 VS Code에서 코드를 고치고 push한 뒤, 여기서 셀 1만 다시 실행하면 된다.


In [ ]:
# 셀 1: 레포 가져오기
# 비공개 레포이므로 Colab 보안 비밀(GH_TOKEN)에 GitHub 토큰을 넣어둬야 한다.
#   왼쪽 열쇠 아이콘 -> 새 보안 비밀 -> 이름 GH_TOKEN -> '노트북 액세스' 켜기
import os
USER, REPO = 'HojuneLee0106', 'komap-2026'
DST = f'/content/{REPO}'

try:
    from google.colab import userdata
    url = f'https://{userdata.get("GH_TOKEN")}@github.com/{USER}/{REPO}.git'
except Exception as e:
    print('토큰 없음, 공개 레포로 간주:', type(e).__name__)
    url = f'https://github.com/{USER}/{REPO}.git'

if os.path.exists(DST):
    !cd {DST} && git pull -q          # 토큰은 출력하지 않는다
else:
    !git clone -q {url} {DST}
%cd {DST}
!pip -q install -r requirements.txt
print('준비 완료:', os.listdir('src'))


In [ ]:
# 셀 2: 데이터 준비 (Drive -> 로컬 SSD, I/O 병목 제거)
from google.colab import drive; drive.mount('/content/drive')
import shutil, os
SRC = '/content/drive/MyDrive/Colab Notebooks'
for s in ['train', 'valid', 'test']:
    d = f'/content/data/{s}'
    if not os.path.exists(d):
        base = f'{SRC}/{s}'
        real = next(r for r, subs, _ in os.walk(base) if 'images' in subs)
        shutil.copytree(real, d)
    print(s, len(os.listdir(f'{d}/images')))


In [ ]:
# 셀 3: 사전학습 가중치 1회 저장 (이후 오프라인 동작)
WD = '/content/drive/MyDrive/komap/weights'
!python -m src.prepare_weights timm-efficientnet-b3 {WD}


In [ ]:
# 셀 4: 기준선 확인 — 모델은 이 숫자를 넘어야 한다
from src.config import CFG
from src import baseline
baseline.main(CFG(), split='valid')


In [ ]:
# 셀 5: 첫 학습 — 목적은 성능이 아니라 '안 터지는지'와 '1 epoch 소요 시간'
from src import train
cfg = CFG(exp='c1_smoke', loss='ce', folds=(0,), epochs=20, val_every=5)
train.main(cfg)


In [ ]:
# 셀 6: 핵심 실험 — 4class 직접 분할 vs 3class + 두께 분리
a = train.main(CFG(exp='c1_4class', mode='4class', loss='wce_dice'))
b = train.main(CFG(exp='c1_3class', mode='3class', loss='wce_dice'))


In [ ]:
# 셀 7: 추론 + 제출물 생성
from src import predict
predict.main(CFG(exp='c1_4class'))


In [ ]:
# (참고) 로컬에서 코드를 고친 뒤 재시작 없이 반영하기
import importlib
for m in ['config','data','model','losses','metrics','postprocess','baseline','train','predict']:
    importlib.reload(importlib.import_module(f'src.{m}'))
